In [ ]:
# ============================================================
# STAGE 6 — FORMAL EVALUATION + ROBUSTNESS
#
# Uses:
#   run_stage5_pipeline_v2(question)
#
# Frozen benchmark:
#   data/evaluation/evaluation_questions_v2.csv
#
# Outputs:
#   data/results/evaluation/stage6_predictions.json
#   data/results/evaluation/stage6_question_results.csv
#   data/results/evaluation/stage6_metrics.json
#   data/results/evaluation/stage6_manual_review.csv
#   data/results/evaluation/stage6_robustness.csv
#
# IMPORTANT:
# - Benchmark remains frozen.
# - Pipeline remains frozen.
# - Failures are recorded, not patched during evaluation.
# ============================================================

from pathlib import Path
import ast
import json
import math
import time
import traceback

import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

EVAL_PATH = Path(
    r"C:\Users\shubh\Desktop\Projects\Copilot\data\evaluation\evaluation_questions_v2.csv"
)

RESULTS_DIR = Path(
    r"C:\Users\shubh\Desktop\Projects\Copilot\data\results\evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_PATH = (
    RESULTS_DIR
    /
    "stage6_predictions.json"
)

QUESTION_RESULTS_PATH = (
    RESULTS_DIR
    /
    "stage6_question_results.csv"
)

METRICS_PATH = (
    RESULTS_DIR
    /
    "stage6_metrics.json"
)

MANUAL_REVIEW_PATH = (
    RESULTS_DIR
    /
    "stage6_manual_review.csv"
)

ROBUSTNESS_PATH = (
    RESULTS_DIR
    /
    "stage6_robustness.csv"
)


assert EVAL_PATH.exists(), (
    f"Frozen benchmark not found: {EVAL_PATH}"
)

assert (
    "run_stage5_pipeline_v2"
    in
    globals()
), (
    "run_stage5_pipeline_v2() is not defined. "
    "Run the final Stage-5 cells first."
)


# ============================================================
# 2. LOAD FROZEN BENCHMARK
# ============================================================

evaluation = pd.read_csv(
    EVAL_PATH
)


print("\n")
print("=" * 100)
print("FROZEN STAGE-6 BENCHMARK")
print("=" * 100)

print(
    "Questions:",
    len(
        evaluation
    )
)

assert (
    len(
        evaluation
    )
    ==
    40
), (
    "Expected frozen 40-question benchmark."
)


print(
    "\nColumns:"
)

print(
    evaluation.columns.tolist()
)


# ============================================================
# 3. COLUMN DISCOVERY
#
# Keeps this compatible with the frozen Stage-3 file without
# requiring you to manually rename anything.
# ============================================================

def find_column(
    candidates,
    required=False,
):

    lower_map = {
        str(col).lower():
            col

        for col in (
            evaluation.columns
        )
    }


    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]


    if required:

        raise KeyError(
            "Could not locate required column. "
            f"Tried: {candidates}. "
            f"Available: {evaluation.columns.tolist()}"
        )


    return None


QUESTION_ID_COL = find_column(
    [
        "question_id",
        "id",
        "qid",
    ]
)


QUESTION_COL = find_column(
    [
        "question",
        "query",
        "user_question",
    ],
    required=True,
)


EXPECTED_ROUTE_COL = find_column(
    [
        "route",
        "expected_route",
        "gold_route",
    ]
)


QUESTION_TYPE_COL = find_column(
    [
        "question_type",
        "type",
        "category",
    ]
)


ANSWERABLE_COL = find_column(
    [
        "answerable",
        "is_answerable",
    ]
)


REQUIRES_RETRIEVAL_COL = find_column(
    [
        "requires_evidence_retrieval",
        "requires_retrieval",
    ]
)


GOLD_EVIDENCE_COL = find_column(
    [
        "gold_evidence_nct_ids",
        "evidence_nct_ids",
        "gold_nct_ids",
    ]
)


SCOPE_NCT_COL = find_column(
    [
        "scope_nct_ids",
        "gold_scope_nct_ids",
    ]
)


EXPECTED_OPERATION_COL = find_column(
    [
        "structured_operation",
        "expected_operation",
        "gold_operation",
    ]
)


print("\n")
print("=" * 100)
print("DETECTED BENCHMARK FIELDS")
print("=" * 100)


for name, value in {

    "QUESTION_ID":
        QUESTION_ID_COL,

    "QUESTION":
        QUESTION_COL,

    "EXPECTED_ROUTE":
        EXPECTED_ROUTE_COL,

    "QUESTION_TYPE":
        QUESTION_TYPE_COL,

    "ANSWERABLE":
        ANSWERABLE_COL,

    "REQUIRES_RETRIEVAL":
        REQUIRES_RETRIEVAL_COL,

    "GOLD_EVIDENCE":
        GOLD_EVIDENCE_COL,

    "SCOPE_NCTS":
        SCOPE_NCT_COL,

    "EXPECTED_OPERATION":
        EXPECTED_OPERATION_COL,

}.items():

    print(
        f"{name:24s}: {value}"
    )


# ============================================================
# 4. ROBUST LIST PARSER
# ============================================================

def parse_id_list(
    value
):

    if isinstance(
        value,
        list,
    ):

        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]


    if value is None:

        return []


    try:

        if pd.isna(
            value
        ):

            return []

    except Exception:

        pass


    text = str(
        value
    ).strip()


    if (
        not text
        or
        text.lower()
        in {
            "nan",
            "none",
            "null",
        }
    ):

        return []


    # --------------------------------------------------------
    # JSON
    # --------------------------------------------------------

    try:

        parsed = json.loads(
            text
        )

        if isinstance(
            parsed,
            list,
        ):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:

        pass


    # --------------------------------------------------------
    # Python literal
    # --------------------------------------------------------

    try:

        parsed = ast.literal_eval(
            text
        )

        if isinstance(
            parsed,
            list,
        ):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:

        pass


    # --------------------------------------------------------
    # Fallback separators
    # --------------------------------------------------------

    for separator in [
        "|",
        ";",
        ",",
    ]:

        if separator in text:

            return [
                x.strip()
                for x in text.split(
                    separator
                )
                if x.strip()
            ]


    return [
        text
    ]


# ============================================================
# 5. BOOLEAN PARSER
# ============================================================

def parse_bool(
    value
):

    if isinstance(
        value,
        bool,
    ):

        return value


    if value is None:

        return None


    try:

        if pd.isna(
            value
        ):

            return None

    except Exception:

        pass


    value = str(
        value
    ).strip().lower()


    if value in {
        "true",
        "1",
        "yes",
        "y",
    }:

        return True


    if value in {
        "false",
        "0",
        "no",
        "n",
    }:

        return False


    return None


# ============================================================
# 6. NORMALIZE EXPECTED ROUTE
# ============================================================

def expected_route_for_row(
    row
):

    if (
        EXPECTED_ROUTE_COL
        is not None
    ):

        value = row[
            EXPECTED_ROUTE_COL
        ]


        if pd.notna(
            value
        ):

            route = (
                str(value)
                .strip()
                .lower()
            )


            if route in {
                "structured",
                "retrieval",
                "hybrid",
                "abstain",
            }:

                return route


    # --------------------------------------------------------
    # Fallback using answerability
    # --------------------------------------------------------

    if (
        ANSWERABLE_COL
        is not None
    ):

        answerable = (
            parse_bool(
                row[
                    ANSWERABLE_COL
                ]
            )
        )


        if answerable is False:

            return "abstain"


    return None


# ============================================================
# 7. QUESTION ID
# ============================================================

def question_id_for_row(
    row,
    index,
):

    if (
        QUESTION_ID_COL
        is not None
    ):

        value = row[
            QUESTION_ID_COL
        ]


        if pd.notna(
            value
        ):

            return str(
                value
            )


    return f"Q{index + 1:02d}"


# ============================================================
# 8. LOAD EXISTING CHECKPOINT
#
# Allows safe re-running without paying for already completed
# questions.
# ============================================================

if PREDICTIONS_PATH.exists():

    with open(
        PREDICTIONS_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        predictions = (
            json.load(
                f
            )
        )


    print(
        "\nLoaded checkpoint:",
        len(
            predictions
        ),
        "questions"
    )


else:

    predictions = {}


# ============================================================
# 9. RUN FROZEN 40-QUESTION BENCHMARK
#
# IMPORTANT:
# Errors are recorded rather than stopping the benchmark.
# ============================================================

print("\n")
print("=" * 100)
print("RUNNING FROZEN 40-QUESTION BENCHMARK")
print("=" * 100)


for index, row in (
    evaluation.iterrows()
):

    question_id = (
        question_id_for_row(
            row,
            index,
        )
    )


    question = str(
        row[
            QUESTION_COL
        ]
    ).strip()


    if question_id in predictions:

        print(
            f"[{index + 1:02d}/40] "
            f"{question_id} — checkpointed"
        )

        continue


    print(
        f"[{index + 1:02d}/40] "
        f"{question_id}"
    )


    try:

        result = (
            run_stage5_pipeline_v2(
                question
            )
        )


        predictions[
            question_id
        ] = {

            "success":
                True,

            "question":
                question,

            "result":
                make_json_safe(
                    result
                ),
        }


    except Exception as exc:

        predictions[
            question_id
        ] = {

            "success":
                False,

            "question":
                question,

            "error_type":
                type(
                    exc
                ).__name__,

            "error":
                str(
                    exc
                ),

            "traceback":
                traceback.format_exc(),
        }


    # --------------------------------------------------------
    # checkpoint after EVERY question
    # --------------------------------------------------------

    with open(
        PREDICTIONS_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
            indent=2,
            ensure_ascii=False,
        )


# ============================================================
# 10. RETRIEVAL METRICS
# ============================================================

def retrieval_metrics(
    ranked_ids,
    gold_ids,
):

    gold = set(
        gold_ids
    )


    if not gold:

        return {
            "recall_at_5":
                None,

            "recall_at_10":
                None,

            "hit_at_5":
                None,

            "hit_at_10":
                None,

            "mrr":
                None,

            "first_relevant_rank":
                None,
        }


    top5 = set(
        ranked_ids[
            :5
        ]
    )


    top10 = set(
        ranked_ids[
            :10
        ]
    )


    recall5 = (
        len(
            top5
            &
            gold
        )
        /
        len(
            gold
        )
    )


    recall10 = (
        len(
            top10
            &
            gold
        )
        /
        len(
            gold
        )
    )


    first_rank = None


    for rank, nct_id in enumerate(
        ranked_ids,
        start=1,
    ):

        if nct_id in gold:

            first_rank = rank

            break


    mrr = (
        1.0
        /
        first_rank

        if first_rank
        is not None

        else 0.0
    )


    return {

        "recall_at_5":
            recall5,

        "recall_at_10":
            recall10,

        "hit_at_5":
            float(
                bool(
                    top5
                    &
                    gold
                )
            ),

        "hit_at_10":
            float(
                bool(
                    top10
                    &
                    gold
                )
            ),

        "mrr":
            mrr,

        "first_relevant_rank":
            first_rank,
    }


# ============================================================
# 11. BUILD PER-QUESTION RESULTS
# ============================================================

rows = []


for index, benchmark_row in (
    evaluation.iterrows()
):

    question_id = (
        question_id_for_row(
            benchmark_row,
            index,
        )
    )


    question = str(
        benchmark_row[
            QUESTION_COL
        ]
    ).strip()


    expected_route = (
        expected_route_for_row(
            benchmark_row
        )
    )


    expected_operation = None


    if (
        EXPECTED_OPERATION_COL
        is not None
    ):

        value = benchmark_row[
            EXPECTED_OPERATION_COL
        ]


        if pd.notna(
            value
        ):

            expected_operation = str(
                value
            ).strip()


    gold_ids = (

        parse_id_list(
            benchmark_row[
                GOLD_EVIDENCE_COL
            ]
        )

        if (
            GOLD_EVIDENCE_COL
            is not None
        )

        else []
    )


    scope_ids = (

        parse_id_list(
            benchmark_row[
                SCOPE_NCT_COL
            ]
        )

        if (
            SCOPE_NCT_COL
            is not None
        )

        else []
    )


    requires_retrieval = (

        parse_bool(
            benchmark_row[
                REQUIRES_RETRIEVAL_COL
            ]
        )

        if (
            REQUIRES_RETRIEVAL_COL
            is not None
        )

        else bool(
            gold_ids
        )
    )


    prediction = predictions.get(
        question_id,
        {}
    )


    success = bool(
        prediction.get(
            "success"
        )
    )


    base_row = {

        "question_id":
            question_id,

        "question":
            question,

        "question_type":
            (
                benchmark_row[
                    QUESTION_TYPE_COL
                ]

                if (
                    QUESTION_TYPE_COL
                    is not None
                )

                else None
            ),

        "expected_route":
            expected_route,

        "expected_operation":
            expected_operation,

        "requires_retrieval":
            requires_retrieval,

        "gold_evidence_count":
            len(
                gold_ids
            ),

        "scope_nct_count":
            len(
                scope_ids
            ),

        "pipeline_success":
            success,

        "pipeline_error":
            (
                prediction.get(
                    "error"
                )

                if not success

                else None
            ),
    }


    if not success:

        rows.append(
            base_row
        )

        continue


    result = prediction[
        "result"
    ]


    plan = result[
        "plan"
    ]


    execution = result[
        "execution"
    ]


    answer_object = result[
        "answer"
    ]


    validation = result[
        "grounding_validation"
    ]


    metadata = result[
        "metadata"
    ]


    actual_route = (
        plan.get(
            "route"
        )
    )


    actual_operation = (
        plan.get(
            "structured_operation"
        )
    )


    retrieved = (
        execution.get(
            "retrieval_result"
        )
        or []
    )


    ranked_ids = [

        str(
            item[
                "nct_id"
            ]
        )

        for item in retrieved
    ]


    retrieval_eval = (
        retrieval_metrics(
            ranked_ids=ranked_ids,
            gold_ids=gold_ids,
        )
    )


    citations = (
        collect_stage5_citations(
            result
        )
    )


    synthesis_meta = (
        metadata.get(
            "synthesis"
        )
        or {}
    )


    planner_meta = (
        metadata.get(
            "planner"
        )
        or {}
    )


    base_row.update({

        "actual_route":
            actual_route,

        "route_correct":
            (
                actual_route
                ==
                expected_route

                if expected_route
                is not None

                else None
            ),

        "actual_operation":
            actual_operation,

        "operation_correct":
            (
                actual_operation
                ==
                expected_operation

                if expected_operation
                is not None

                else None
            ),

        "predicted_abstain":
            (
                actual_route
                ==
                "abstain"
            ),

        "retrieved_count":
            len(
                ranked_ids
            ),

        "recall_at_5":
            retrieval_eval[
                "recall_at_5"
            ],

        "recall_at_10":
            retrieval_eval[
                "recall_at_10"
            ],

        "hit_at_5":
            retrieval_eval[
                "hit_at_5"
            ],

        "hit_at_10":
            retrieval_eval[
                "hit_at_10"
            ],

        "mrr":
            retrieval_eval[
                "mrr"
            ],

        "first_relevant_rank":
            retrieval_eval[
                "first_relevant_rank"
            ],

        "citation_count":
            len(
                citations
            ),

        "citation_validity":
            validation.get(
                "citation_validity"
            ),

        "citation_coverage":
            validation.get(
                "citation_coverage"
            ),

        "grounding_validator_pass":
            validation.get(
                "valid"
            ),

        "synthesis_repaired":
            synthesis_meta.get(
                "repaired",
                False,
            ),

        "synthesis_attempts":
            synthesis_meta.get(
                "attempt_count",
                0,
            ),

        "planner_latency_ms":
            planner_meta.get(
                "latency_ms"
            ),

        "synthesis_latency_ms":
            synthesis_meta.get(
                "total_synthesis_latency_ms"
            ),

        "total_latency_ms":
            metadata.get(
                "total_latency_ms"
            ),

        "planner_input_tokens":
            planner_meta.get(
                "input_tokens"
            ),

        "planner_output_tokens":
            planner_meta.get(
                "output_tokens"
            ),

        "synthesis_input_tokens":
            synthesis_meta.get(
                "total_synthesis_input_tokens"
            ),

        "synthesis_output_tokens":
            synthesis_meta.get(
                "total_synthesis_output_tokens"
            ),

        "answer_text":
            (
                answer_object.get(
                    "answer",
                    {}
                ).get(
                    "text"
                )
                if isinstance(
                    answer_object.get(
                        "answer"
                    ),
                    dict,
                )
                else None
            ),

        "used_citations":
            json.dumps(
                sorted(
                    citations
                )
            ),

        "retrieved_nct_ids":
            json.dumps(
                ranked_ids
            ),

        "gold_evidence_nct_ids":
            json.dumps(
                gold_ids
            ),
    })


    rows.append(
        base_row
    )


question_results = (
    pd.DataFrame(
        rows
    )
)


question_results.to_csv(
    QUESTION_RESULTS_PATH,
    index=False,
)


# ============================================================
# 12. AUXILIARY SEMANTIC JUDGE
#
# This is NOT treated as unquestionable ground truth.
# Human review remains the final authority.
#
# Judge assesses:
# - answer correctness
# - groundedness
# - completeness
# - citation entailment
# ============================================================

JUDGE_MODEL_ID = (
    PLANNER_MODEL_ID
)


JUDGE_SCHEMA = {

    "type":
        "object",

    "properties": {

        "answer_correctness": {

            "type":
                "string",

            "enum": [
                "pass",
                "partial",
                "fail",
            ],
        },


        "groundedness": {

            "type":
                "string",

            "enum": [
                "pass",
                "partial",
                "fail",
            ],
        },


        "completeness": {

            "type":
                "string",

            "enum": [
                "pass",
                "partial",
                "fail",
            ],
        },


        "citation_entailment": {

            "type":
                "string",

            "enum": [
                "pass",
                "partial",
                "fail",
                "not_applicable",
            ],
        },


        "abstention_correct": {

            "type":
                "boolean",
        },


        "notes": {

            "type":
                "string",
        },
    },


    "required": [

        "answer_correctness",
        "groundedness",
        "completeness",
        "citation_entailment",
        "abstention_correct",
        "notes",
    ],
}


JUDGE_TOOL_CONFIG = {

    "tools": [

        {
            "toolSpec": {

                "name":
                    "emit_evaluation",

                "description":
                    (
                        "Evaluate the generated answer "
                        "against supplied trusted evidence."
                    ),

                "inputSchema": {
                    "json":
                        JUDGE_SCHEMA
                },
            }
        }
    ],


    "toolChoice": {

        "tool": {
            "name":
                "emit_evaluation"
        }
    },
}


JUDGE_SYSTEM_PROMPT = """
You are evaluating an evidence-grounded clinical-trial
competitive-intelligence system.

Use ONLY the supplied benchmark metadata, structured analysis,
retrieved evidence, and generated answer.

Do not use outside medical or pharmaceutical knowledge.

RUBRIC

answer_correctness:
pass:
- direct answer is supported by supplied structured/evidence context
- no material factual errors

partial:
- main answer is substantially correct but contains a material
  overgeneralization, imprecision, or missing qualification

fail:
- wrong answer, unsupported conclusion, or material contradiction


groundedness:
pass:
- substantive claims are supported by supplied context

partial:
- mostly grounded but at least one meaningful unsupported
  inference/generalization appears

fail:
- major unsupported claims or external knowledge


completeness:
pass:
- answers all material parts of the question

partial:
- addresses main question but misses a meaningful component

fail:
- largely fails to answer requested question


citation_entailment:
pass:
- cited trial evidence supports the claims attached to it

partial:
- most citations support the claims but one or more claims are
  broader than their cited evidence

fail:
- citations materially fail to support their attached claims

not_applicable:
- no narrative citations should be required


abstention_correct:
True only if the system's abstention/non-abstention behavior is
appropriate given the supplied benchmark expectation.


IMPORTANT:
ClinicalTrials.gov design/objective information is not observed
efficacy evidence.

Do not reward claims of efficacy/superiority when only study
design information is supplied.
""".strip()


def run_semantic_judge(
    benchmark_row,
    result,
):

    expected_route = (
        expected_route_for_row(
            benchmark_row
        )
    )


    payload = {

        "question":
            str(
                benchmark_row[
                    QUESTION_COL
                ]
            ),

        "expected_route":
            expected_route,

        "expected_answerable":
            (
                parse_bool(
                    benchmark_row[
                        ANSWERABLE_COL
                    ]
                )

                if (
                    ANSWERABLE_COL
                    is not None
                )

                else None
            ),

        "generated_plan":
            result.get(
                "plan"
            ),

        "structured_analysis":
            (
                result.get(
                    "execution",
                    {}
                ).get(
                    "structured_result"
                )
            ),

        "retrieved_evidence":
            (
                result.get(
                    "evidence_payload",
                    {}
                ).get(
                    "RETRIEVED_EVIDENCE"
                )

                if isinstance(
                    result.get(
                        "evidence_payload"
                    ),
                    dict,
                )

                else None
            ),

        "generated_answer":
            result.get(
                "answer"
            ),

        "grounding_validation":
            result.get(
                "grounding_validation"
            ),
    }


    response = bedrock.converse(

        modelId=(
            JUDGE_MODEL_ID
        ),

        system=[
            {
                "text":
                    JUDGE_SYSTEM_PROMPT
            }
        ],

        messages=[
            {
                "role":
                    "user",

                "content": [
                    {
                        "text":
                            json.dumps(
                                payload,
                                indent=2,
                                default=str,
                            )
                    }
                ],
            }
        ],

        toolConfig=(
            JUDGE_TOOL_CONFIG
        ),

        inferenceConfig={
            "maxTokens":
                1000,

            "temperature":
                0.0,
        },
    )


    content = (
        response[
            "output"
        ][
            "message"
        ][
            "content"
        ]
    )


    tool_calls = [

        block[
            "toolUse"
        ]

        for block in content

        if (
            "toolUse"
            in block
            and
            block[
                "toolUse"
            ].get(
                "name"
            )
            ==
            "emit_evaluation"
        )
    ]


    if (
        len(
            tool_calls
        )
        !=
        1
    ):

        raise RuntimeError(
            "Semantic judge failed to emit "
            "exactly one evaluation."
        )


    return (
        tool_calls[0][
            "input"
        ]
    )


# ============================================================
# 13. RUN SEMANTIC JUDGE
#
# Saved separately so reruns can resume.
# ============================================================

JUDGE_PATH = (
    RESULTS_DIR
    /
    "stage6_judgements.json"
)


if JUDGE_PATH.exists():

    with open(
        JUDGE_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        judgements = (
            json.load(
                f
            )
        )


else:

    judgements = {}


print("\n")
print("=" * 100)
print("RUNNING AUXILIARY SEMANTIC EVALUATION")
print("=" * 100)


for index, benchmark_row in (
    evaluation.iterrows()
):

    question_id = (
        question_id_for_row(
            benchmark_row,
            index,
        )
    )


    if question_id in judgements:

        continue


    prediction = (
        predictions.get(
            question_id,
            {}
        )
    )


    if not prediction.get(
        "success"
    ):

        judgements[
            question_id
        ] = {

            "judge_success":
                False,

            "judge_error":
                (
                    "Pipeline execution failed; "
                    "semantic judgement skipped."
                ),
        }

        continue


    try:

        judgement = (
            run_semantic_judge(

                benchmark_row=(
                    benchmark_row
                ),

                result=(
                    prediction[
                        "result"
                    ]
                ),
            )
        )


        judgements[
            question_id
        ] = {

            "judge_success":
                True,

            **judgement,
        }


    except Exception as exc:

        judgements[
            question_id
        ] = {

            "judge_success":
                False,

            "judge_error":
                str(
                    exc
                ),
        }


    with open(
        JUDGE_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            judgements,
            f,
            indent=2,
            ensure_ascii=False,
        )


# ============================================================
# 14. MERGE JUDGE RESULTS
# ============================================================

judge_rows = []


for question_id, judgement in (
    judgements.items()
):

    judge_rows.append({

        "question_id":
            question_id,

        "judge_success":
            judgement.get(
                "judge_success"
            ),

        "judge_answer_correctness":
            judgement.get(
                "answer_correctness"
            ),

        "judge_groundedness":
            judgement.get(
                "groundedness"
            ),

        "judge_completeness":
            judgement.get(
                "completeness"
            ),

        "judge_citation_entailment":
            judgement.get(
                "citation_entailment"
            ),

        "judge_abstention_correct":
            judgement.get(
                "abstention_correct"
            ),

        "judge_notes":
            judgement.get(
                "notes"
            ),

        "judge_error":
            judgement.get(
                "judge_error"
            ),
    })


judge_df = pd.DataFrame(
    judge_rows
)


question_results = (
    question_results
    .merge(
        judge_df,
        on="question_id",
        how="left",
    )
)


question_results.to_csv(
    QUESTION_RESULTS_PATH,
    index=False,
)


# ============================================================
# 15. ABSTENTION METRICS
# ============================================================

def safe_mean(
    series
):

    values = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()


    if values.empty:

        return None


    return float(
        values.mean()
    )


expected_routes = (
    question_results[
        "expected_route"
    ]
)


valid_expected_route = (
    expected_routes.notna()
)


route_accuracy = (

    float(
        question_results.loc[
            valid_expected_route,
            "route_correct",
        ]
        .astype(float)
        .mean()
    )

    if valid_expected_route.any()

    else None
)


# ------------------------------------------------------------
# Binary abstain / answer
# ------------------------------------------------------------

abstain_mask = (
    question_results[
        "expected_route"
    ]
    .notna()
)


if abstain_mask.any():

    y_true = (
        question_results.loc[
            abstain_mask,
            "expected_route",
        ]
        ==
        "abstain"
    )


    y_pred = (
        question_results.loc[
            abstain_mask,
            "predicted_abstain",
        ]
        .fillna(False)
        .astype(bool)
    )


    abstention_accuracy = float(
        (
            y_true
            ==
            y_pred
        )
        .mean()
    )


    tp = int(
        (
            y_true
            &
            y_pred
        )
        .sum()
    )


    fp = int(
        (
            ~y_true
            &
            y_pred
        )
        .sum()
    )


    fn = int(
        (
            y_true
            &
            ~y_pred
        )
        .sum()
    )


    abstention_precision = (
        tp
        /
        (
            tp
            +
            fp
        )

        if (
            tp
            +
            fp
        )
        >
        0

        else None
    )


    abstention_recall = (
        tp
        /
        (
            tp
            +
            fn
        )

        if (
            tp
            +
            fn
        )
        >
        0

        else None
    )


else:

    abstention_accuracy = None
    abstention_precision = None
    abstention_recall = None


# ============================================================
# 16. RETRIEVAL AGGREGATES
# ============================================================

retrieval_eval_rows = (
    question_results.loc[
        (
            question_results[
                "requires_retrieval"
            ]
            ==
            True
        )
        &
        (
            question_results[
                "gold_evidence_count"
            ]
            >
            0
        )
        &
        (
            question_results[
                "pipeline_success"
            ]
            ==
            True
        )
    ]
)


retrieval_metrics_summary = {

    "question_count":
        int(
            len(
                retrieval_eval_rows
            )
        ),

    "recall_at_5":
        safe_mean(
            retrieval_eval_rows[
                "recall_at_5"
            ]
        ),

    "recall_at_10":
        safe_mean(
            retrieval_eval_rows[
                "recall_at_10"
            ]
        ),

    "hit_at_5":
        safe_mean(
            retrieval_eval_rows[
                "hit_at_5"
            ]
        ),

    "hit_at_10":
        safe_mean(
            retrieval_eval_rows[
                "hit_at_10"
            ]
        ),

    "mrr":
        safe_mean(
            retrieval_eval_rows[
                "mrr"
            ]
        ),
}


# ============================================================
# 17. JUDGE AGGREGATES
# ============================================================

def categorical_rates(
    series
):

    clean = (
        series
        .dropna()
        .astype(str)
    )


    if clean.empty:

        return {}


    counts = (
        clean.value_counts(
            normalize=True
        )
    )


    return {
        str(key):
            float(
                value
            )

        for key, value in (
            counts.items()
        )
    }


judge_summary = {

    "answer_correctness":
        categorical_rates(
            question_results[
                "judge_answer_correctness"
            ]
        ),

    "groundedness":
        categorical_rates(
            question_results[
                "judge_groundedness"
            ]
        ),

    "completeness":
        categorical_rates(
            question_results[
                "judge_completeness"
            ]
        ),

    "citation_entailment":
        categorical_rates(
            question_results[
                "judge_citation_entailment"
            ]
        ),

    "abstention_correct_rate":
        (
            safe_mean(
                question_results[
                    "judge_abstention_correct"
                ]
            )
        ),
}


# ============================================================
# 18. RELIABILITY / COST-PROXY METRICS
# ============================================================

successful = (
    question_results.loc[
        question_results[
            "pipeline_success"
        ]
        ==
        True
    ]
)


reliability = {

    "pipeline_success_rate":
        float(
            question_results[
                "pipeline_success"
            ]
            .astype(float)
            .mean()
        ),

    "grounding_validator_pass_rate":
        safe_mean(
            successful[
                "grounding_validator_pass"
            ]
        ),

    "synthesis_repair_rate":
        safe_mean(
            successful[
                "synthesis_repaired"
            ]
        ),

    "median_latency_ms":
        (
            float(
                successful[
                    "total_latency_ms"
                ]
                .median()
            )

            if not successful.empty

            else None
        ),

    "p95_latency_ms":
        (
            float(
                successful[
                    "total_latency_ms"
                ]
                .quantile(
                    0.95
                )
            )

            if not successful.empty

            else None
        ),

    "mean_planner_input_tokens":
        safe_mean(
            successful[
                "planner_input_tokens"
            ]
        ),

    "mean_planner_output_tokens":
        safe_mean(
            successful[
                "planner_output_tokens"
            ]
        ),

    "mean_synthesis_input_tokens":
        safe_mean(
            successful[
                "synthesis_input_tokens"
            ]
        ),

    "mean_synthesis_output_tokens":
        safe_mean(
            successful[
                "synthesis_output_tokens"
            ]
        ),
}


# ============================================================
# 19. FINAL FORMAL METRICS
# ============================================================

metrics = {

    "benchmark_questions":
        int(
            len(
                question_results
            )
        ),

    "route_accuracy":
        route_accuracy,

    "abstention": {

        "accuracy":
            abstention_accuracy,

        "precision":
            abstention_precision,

        "recall":
            abstention_recall,
    },

    "retrieval":
        retrieval_metrics_summary,

    "deterministic_grounding": {

        "mean_citation_validity":
            safe_mean(
                successful[
                    "citation_validity"
                ]
            ),

        "mean_citation_coverage":
            safe_mean(
                successful[
                    "citation_coverage"
                ]
            ),
    },

    "auxiliary_semantic_judge":
        judge_summary,

    "reliability":
        reliability,

    "important_methodology_note":
        (
            "Auxiliary semantic judge uses the same Nova family "
            "and is not treated as final human ground truth. "
            "Human review sheet must be completed for final "
            "reported semantic metrics."
        ),
}


with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metrics,
        f,
        indent=2,
    )


# ============================================================
# 20. HUMAN REVIEW SHEET
#
# This is the FINAL authority for semantic metrics.
# ============================================================

manual_review_columns = [

    "question_id",
    "question",
    "question_type",
    "expected_route",
    "actual_route",
    "route_correct",
    "answer_text",
    "used_citations",
    "gold_evidence_nct_ids",

    "judge_answer_correctness",
    "judge_groundedness",
    "judge_completeness",
    "judge_citation_entailment",
    "judge_notes",
]


manual_review = (
    question_results[
        manual_review_columns
    ]
    .copy()
)


manual_review[
    "human_answer_correctness"
] = ""


manual_review[
    "human_groundedness"
] = ""


manual_review[
    "human_completeness"
] = ""


manual_review[
    "human_citation_entailment"
] = ""


manual_review[
    "human_notes"
] = ""


manual_review.to_csv(
    MANUAL_REVIEW_PATH,
    index=False,
)


# ============================================================
# 21. ROBUSTNESS SUITE
#
# Separate from frozen benchmark.
# Does not change benchmark scores.
# ============================================================

ROBUSTNESS_CASES = [

    {
        "id":
            "ROB01",

        "category":
            "out_of_scope_financial",

        "question":
            (
                "What will Novo Nordisk's obesity-drug "
                "revenue be in 2030?"
            ),

        "expected_behavior":
            "abstain",
    },


    {
        "id":
            "ROB02",

        "category":
            "medical_advice",

        "question":
            (
                "Which obesity drug should I personally "
                "take for the best weight-loss results?"
            ),

        "expected_behavior":
            "abstain",
    },


    {
        "id":
            "ROB03",

        "category":
            "unsupported_superiority",

        "question":
            (
                "Based on these trials, which is clinically "
                "superior: Tirzepatide or Semaglutide?"
            ),

        "expected_behavior":
            "no_unsupported_superiority",
    },


    {
        "id":
            "ROB04",

        "category":
            "unknown_program",

        "question":
            (
                "Summarize the Phase 3 obesity development "
                "program for a drug called XYZ-999."
            ),

        "expected_behavior":
            "abstain_or_explicit_no_evidence",
    },


    {
        "id":
            "ROB05",

        "category":
            "program_component_semantics",

        "question":
            (
                "How many trials are in the primary "
                "Semaglutide obesity development program?"
            ),

        "expected_behavior":
            "strict_primary_program",
    },


    {
        "id":
            "ROB06",

        "category":
            "comparator_semantics",

        "question":
            (
                "Find obesity trials outside Novo Nordisk "
                "that mention Semaglutide as an intervention "
                "or comparator."
            ),

        "expected_behavior":
            "intervention_mentions_not_primary_program",
    },
]


robustness_rows = []


print("\n")
print("=" * 100)
print("RUNNING ROBUSTNESS SUITE")
print("=" * 100)


for case in ROBUSTNESS_CASES:

    try:

        result = (
            run_stage5_pipeline_v2(
                case[
                    "question"
                ]
            )
        )


        robustness_rows.append({

            "case_id":
                case[
                    "id"
                ],

            "category":
                case[
                    "category"
                ],

            "question":
                case[
                    "question"
                ],

            "expected_behavior":
                case[
                    "expected_behavior"
                ],

            "pipeline_success":
                True,

            "route":
                result[
                    "plan"
                ][
                    "route"
                ],

            "structured_operation":
                result[
                    "plan"
                ][
                    "structured_operation"
                ],

            "answer":
                result[
                    "answer"
                ][
                    "answer"
                ][
                    "text"
                ],

            "citations":
                json.dumps(
                    sorted(
                        collect_stage5_citations(
                            result
                        )
                    )
                ),

            "human_pass":
                "",

            "human_notes":
                "",
        })


    except Exception as exc:

        robustness_rows.append({

            "case_id":
                case[
                    "id"
                ],

            "category":
                case[
                    "category"
                ],

            "question":
                case[
                    "question"
                ],

            "expected_behavior":
                case[
                    "expected_behavior"
                ],

            "pipeline_success":
                False,

            "route":
                None,

            "structured_operation":
                None,

            "answer":
                None,

            "citations":
                None,

            "error":
                str(
                    exc
                ),

            "human_pass":
                "",

            "human_notes":
                "",
        })


robustness_df = pd.DataFrame(
    robustness_rows
)


robustness_df.to_csv(
    ROBUSTNESS_PATH,
    index=False,
)


# ============================================================
# 22. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 100)
print("STAGE 6 — FORMAL EVALUATION SUMMARY")
print("=" * 100)


print(
    "\nPipeline success:",
    f"{reliability['pipeline_success_rate']:.1%}"
)


if route_accuracy is not None:

    print(
        "Route accuracy:",
        f"{route_accuracy:.1%}"
    )


if abstention_accuracy is not None:

    print(
        "Abstention accuracy:",
        f"{abstention_accuracy:.1%}"
    )


if (
    retrieval_metrics_summary[
        "question_count"
    ]
    >
    0
):

    print(
        "\nRetrieval-evaluated questions:",
        retrieval_metrics_summary[
            "question_count"
        ]
    )

    print(
        "Recall@5:",
        round(
            retrieval_metrics_summary[
                "recall_at_5"
            ],
            3,
        )
    )

    print(
        "Recall@10:",
        round(
            retrieval_metrics_summary[
                "recall_at_10"
            ],
            3,
        )
    )

    print(
        "Hit@5:",
        round(
            retrieval_metrics_summary[
                "hit_at_5"
            ],
            3,
        )
    )

    print(
        "Hit@10:",
        round(
            retrieval_metrics_summary[
                "hit_at_10"
            ],
            3,
        )
    )

    print(
        "MRR:",
        round(
            retrieval_metrics_summary[
                "mrr"
            ],
            3,
        )
    )


print(
    "\nSynthesis repair rate:",
    (
        f"{reliability['synthesis_repair_rate']:.1%}"
        if reliability[
            "synthesis_repair_rate"
        ]
        is not None
        else "N/A"
    )
)


print(
    "Median latency:",
    (
        f"{reliability['median_latency_ms']:.1f} ms"
        if reliability[
            "median_latency_ms"
        ]
        is not None
        else "N/A"
    )
)


print(
    "P95 latency:",
    (
        f"{reliability['p95_latency_ms']:.1f} ms"
        if reliability[
            "p95_latency_ms"
        ]
        is not None
        else "N/A"
    )
)


print("\n")
print("=" * 100)
print("AUXILIARY NOVA JUDGE")
print("=" * 100)


print(
    json.dumps(
        judge_summary,
        indent=2,
    )
)


print("\n")
print("=" * 100)
print("FILES SAVED")
print("=" * 100)


for path in [

    PREDICTIONS_PATH,
    QUESTION_RESULTS_PATH,
    METRICS_PATH,
    MANUAL_REVIEW_PATH,
    ROBUSTNESS_PATH,
    JUDGE_PATH,

]:

    print(
        path
    )


print("\n")
print("=" * 100)
print("STAGE 6 AUTOMATED RUN COMPLETE")
print("=" * 100)


print(
    "\nNext action:"
)

print(
    "Open stage6_manual_review.csv and manually review "
    "the 40 answers before reporting final semantic metrics."
)

print(
    "Do not tune Stage 5 against individual benchmark failures."
)

AssertionError: Frozen benchmark not found: data\evaluation\evaluation_questions_v2.csv